# Model Predictions for 2025 and 2026
## Erica Keklak | 2025-07-08 to present
----
The following contains code snippets that recreate the results of the final models that were the most performant (involving the inevitable trade-off between accuracy and ability to fit a generalized environment) as well as unique code that is used to predict the results of the General Spread Risk Model for what is to be expected for 2025 and 2026.

In [119]:
# Import libraries and packages
## Packages and libraries for data processing (cleaning, preparation, and getting basic statistics from them)

import numpy as np
import pandas as pd
import math as m
import random as rand
import pickle

## Packages and libraries for primarily statistical data visualization

import matplotlib.pyplot as plt
import seaborn as sn

## Packages and libraries for primarily geospatial data visualization

import geopandas as gpd # originally had to try installing to my device using the Anaconda Powershell Prompt command', '                    # conda install -c conda-forge geopandas
from shapely.geometry import Point

## Packages, functions, and libraries for model development

from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split # for test splitting and model verification
from sklearn.ensemble import RandomForestClassifier # for a random forest classification method
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler # encoding is needed to properly process data by algorithms
from sklearn.tree import DecisionTreeClassifier # another decision tree method
from sklearn.metrics import classification_report, confusion_matrix # getting a report from a classification model

In [121]:
# Read basic geometry and final data

county_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/county_geometry.shp')
state_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/state_geometry.shp')
general_county_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Training/general_county_training.shp')
general_state_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Training/general_state_training.shp')
# final_county_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_training.shp')
# final_state_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_training.shp')

# Create 2025 and 2026 rows used in the prediction of their results, obviously with the target data missing and only data present in these rows being
# immutable facts about each county and state

counties_AL = ['Autauga', 'Baldwin', 'Barbour', 'Bibb', 'Blount', 'Bullock', 'Butler', 'Calhoun', 'Chambers', 'Cherokee', 'Chilton', 'Choctaw', 'Clarke',
               'Clay', 'Cleburne', 'Coffee', 'Colbert', 'Conecuh', 'Coosa', 'Covington', 'Crenshaw', 'Cullman', 'Dale', 'Dallas', 'DeKalb', 'Elmore',
               'Escambia', 'Etowah', 'Fayette', 'Franklin', 'Geneva', 'Greene', 'Hale', 'Henry', 'Houston', 'Jackson', 'Jefferson', 'Lamar', 'Lauderdale',
               'Lawrence', 'Lee', 'Limestone', 'Lowndes', 'Macon', 'Madison', 'Marengo', 'Marion', 'Marshall', 'Mobile', 'Monroe', 'Montgomery', 'Morgan',
               'Perry', 'Pickens', 'Pike', 'Randolph', 'Russell', 'St. Clair', 'Shelby', 'Sumter', 'Talladega', 'Tallapoosa', 'Tuscaloosa', 'Walker',
               'Washington', 'Wilcox', 'Winston']
counties_AZ = ['Apache', 'Cochise', 'Coconino', 'Gila', 'Graham', 'Greenlee', 'La Paz', 'Maricopa', 'Mohave', 'Navajo', 'Pima', 'Pinal', 'Santa Cruz',
               'Yavapai', 'Yuma']
counties_AR = ['Arkansas', 'Ashley', 'Baxter', 'Benton', 'Boone', 'Bradley', 'Calhoun', 'Carroll', 'Chicot', 'Clark', 'Clay', 'Cleburne', 'Cleveland',
               'Columbia', 'Conway', 'Craighead', 'Crawford', 'Crittenden', 'Cross', 'Dallas', 'Desha', 'Drew', 'Faulkner', 'Franklin', 'Fulton',
               'Garland', 'Grant', 'Greene', 'Hempstead', 'Hot Spring', 'Howard', 'Independence', 'Izard', 'Jackson', 'Jefferson', 'Johnson', 'Lafayette',
               'Lawrence', 'Lee', 'Lincoln', 'Little River', 'Logan', 'Lonoke', 'Madison', 'Marion', 'Miller', 'Mississippi', 'Monroe', 'Montgomery',
               'Nevada', 'Newton', 'Ouachita', 'Perry', 'Phillips', 'Pike', 'Poinsett', 'Polk', 'Pope', 'Prairie', 'Pulaski', 'Randolph', 'St. Francis',
               'Saline', 'Scott', 'Searcy', 'Sebastian', 'Sevier', 'Sharp', 'Stone', 'Union', 'Van Buren', 'Washington', 'White', 'Woodruff', 'Yell']
counties_CA = ['Alameda', 'Alpine', 'Amador', 'Butte', 'Calaveras', 'Colusa', 'Contra Costa', 'Del Norte', 'El Dorado', 'Fresno', 'Glenn', 'Humboldt',
               'Imperial', 'Inyo', 'Kern', 'Kings', 'Lake', 'Lassen', 'Los Angeles', 'Madera', 'Marin', 'Mariposa', 'Mendocino', 'Merced', 'Modoc',
               'Mono', 'Monterey', 'Napa', 'Nevada', 'Orange', 'Placer', 'Plumas', 'Riverside', 'Sacramento', 'San Benito', 'San Bernardino', 'San Diego',
               'San Francisco', 'San Joaquin', 'San Luis Obispo', 'San Mateo', 'Santa Barbara', 'Santa Clara', 'Santa Cruz', 'Shasta', 'Sierra',
               'Siskiyou', 'Solano', 'Sonoma', 'Stanislaus', 'Sutter', 'Tehama', 'Trinity', 'Tulare', 'Tuolumne', 'Ventura', 'Yolo', 'Yuba']
counties_CO = ['Adams', 'Alamosa', 'Arapahoe', 'Archuleta', 'Baca', 'Bent', 'Boulder', 'Broomfield', 'Chaffee', 'Cheyenne', 'Clear Creek', 'Conejos',
               'Costilla', 'Crowley', 'Custer', 'Delta', 'Denver', 'Dolores', 'Douglas', 'Eagle', 'Elbert', 'El Paso', 'Fremont', 'Garfield', 'Gilpin',
               'Grand', 'Gunnison', 'Hinsdale', 'Huerfano', 'Jackson', 'Jefferson', 'Kiowa', 'Kit Carson', 'Lake', 'La Plata', 'Larimer', 'Las Animas',
               'Lincoln', 'Logan', 'Mesa', 'Mineral', 'Moffat', 'Montezuma', 'Montrose', 'Morgan', 'Otero', 'Ouray', 'Park', 'Phillips', 'Pitkin',
               'Prowers', 'Pueblo', 'Rio Blanco', 'Rio Gande', 'Routt', 'Saguache', 'San Juan', 'San Miguel', 'Sedgwick', 'Summit', 'Teller',
               'Washington', 'Weld', 'Yuma']
counties_CT = ['Fairfield', 'Hartford', 'Litchfield', 'Middlesex', 'New Haven', 'New London', 'Tolland', 'Windham']
counties_DE = ['Kent', 'New Castle', 'Sussex']
counties_FL = ['Alachua', 'Baker', 'Bay', 'Bradford', 'Brevard', 'Broward', 'Calhoun', 'Charlotte', 'Citrus', 'Clay', 'Collier', 'Columbia', 'DeSoto',
               'Dixie', 'Duval', 'Escambia', 'Flagler', 'Franklin', 'Gadsden', 'Gilchrist', 'Glades', 'Gulf', 'Hamilton', 'Hardee', 'Hendry',
               'Hernando', 'Highlands', 'Hillsborough', 'Holmes', 'Indian River', 'Jackson', 'Jefferson', 'Lafayette', 'Lake', 'Lee', 'Leon', 'Levy',
               'Liberty', 'Madison', 'Manatee', 'Marion', 'Martin', 'Miami-Dade', 'Monroe', 'Nassau', 'Okaloosa', 'Okeechobee', 'Orange', 'Osecola',
               'Palm Beach', 'Pasco', 'Pinellas', 'Polk', 'Putnam', 'St. Johns', 'St. Lucie', 'Santa Rosa', 'Sarasota', 'Seminole', 'Sumter', 'Suwannee',
               'Taylor', 'Union', 'Volusia', 'Wakulla', 'Walton', 'Washington']
counties_GA = ['Appling', 'Atkinson', 'Bacon', 'Baker', 'Baldwin', 'Banks', 'Barrow', 'Bartow', 'Ben Hill', 'Berrien', 'Bibb', 'Bleckley', 'Brantley',
               'Brooks', 'Bryan', 'Bulloch', 'Burke', 'Butts', 'Calhoun', 'Camden', 'Candler', 'Carroll', 'Catoosa', 'Charlton', 'Chatham',
               'Chattahoochee', 'Chattooga', 'Cherokee', 'Clarke', 'Clay', 'Clayton', 'Clinch', 'Cobb', 'Coffee', 'Colquitt', 'Columbia', 'Cook',
               'Coweta', 'Crawford', 'Crisp', 'Dade', 'Dawson', 'Decatur', 'DeKalb', 'Dodge', 'Dooly', 'Dougherty', 'Douglas', 'Early', 'Echols',
               'Effingham', 'Elbert', 'Emanuel', 'Evans', 'Fannin', 'Fayette', 'Floyd', 'Forsyth', 'Franklin', 'Fulton', 'Gilmer', 'Glascock', 'Glynn',
               'Gordon', 'Grady', 'Greene', 'Gwinnett', 'Habersham', 'Hall', 'Hancock', 'Haralson', 'Harris', 'Hart', 'Heard', 'Henry', 'Houston',
               'Irwin', 'Jackson', 'Jasper', 'Jeff Davis', 'Jefferson', 'Jenkins', 'Johnson', 'Jones', 'Lamar', 'Lanier', 'Laurens', 'Lee', 'Liberty',
               'Lincoln', 'Long', 'Lowndes', 'Lumpkin', 'Macon', 'Madison', 'Marion', 'McDuffie', 'McIntosh', 'Meriwether', 'Miller', 'Mitchell',
               'Monroe', 'Montgomery', 'Morgan', 'Murray', 'Muscogee', 'Newton', 'Oconew', 'Oglethorpe', 'Paulding', 'Peach', 'Pickens', 'Pierce',
               'Pike', 'Polk', 'Pulaski', 'Putnan', 'Quitman', 'Rabun', 'Randolph', 'Richmond', 'Rockdale', 'Schley', 'Screven', 'Seminole', 'Spalding',
               'Stephens', 'Stewart', 'Sumter', 'Talbot', 'Taliaferro', 'Tattnall', 'Taylor', 'Telfair', 'Terrell', 'Thomas', 'Tift', 'Toombs', 'Towns',
               'Treutlen', 'Troup', 'Turner', 'Twiggs', 'Union', 'Upson', 'Walker', 'Walton', 'Ware', 'Warren', 'Washington', 'Wayne', 'Webster',
               'Wheeler', 'White', 'Whitfield', 'Wilcox', 'Wilkes', 'Wilkinson', 'Worth']
counties_ID = ['Ada', 'Adams', 'Bannock', 'Bear Lake', 'Benewah', 'Bingham', 'Blaine', 'Boise', 'Bonner', 'Bonneville', 'Boundary', 'Butte', 'Camas',
               'Canyon', 'Caribou', 'Cassia', 'Clark', 'Clearwater', 'Custer', 'Elmore', 'Franklin', 'Fremont', 'Gem', 'Gooding', 'Idaho', 'Jefferson',
               'Jerome', 'Kottenai', 'Latah', 'Lemhi', 'Lewis', 'Lincoln', 'Madison', 'Minidoka', 'Nez Perce', 'Oneida', 'Owyhee', 'Payette', 'Power',
               'Shoshone', 'Teton', 'Twin Falls', 'Valley', 'Washington']
counties_IL = ['Adams', 'Alexander', 'Bond', 'Boone', 'Brown', 'Bureau', 'Calhoun', 'Carroll', 'Cass', 'Champaign', 'Christian', 'Clark', 'Clay',
               'Clinton', 'Coles', 'Cook', 'Crawford', 'Cumberland', 'DeKalb', 'DeWitt', 'Douglas', 'DuPage', 'Edgar', 'Edwards', 'Effingham', 'Fayette',
               'Ford', 'Franklin', 'Fulton', 'Gallatin', 'Greene', 'Grundy', 'Hamilton', 'Hancock', 'Hardin', 'Henderson', 'Henry', 'Iroquois', 'Jackson',
               'Jasper', 'Jefferson', 'Jersey', 'Jo Daviess', 'Johnson', 'Kane', 'Kankakee', 'Kendall', 'Knox', 'Lake', 'LaSalle', 'Lawrence', 'Lee',
               'Livingston', 'Logan', 'Macon', 'Macoupin', 'Madison', 'Marion', 'Marshall', 'Mason', 'Massac', 'McDonough', 'McHenry', 'McLean', 'Menard',
               'Mercer', 'Monroe', 'Montgomery', 'Morgan', 'Moultrie', 'Ogle', 'Peoria', 'Perry', 'Piatt', 'Pike', 'Pope', 'Pulaski', 'Putnam', 'Randolph',
               'Richland', 'Rock Island', 'St. Clair', 'Saline', 'Sangamon', 'Schuyler', 'Scott', 'Shelby', 'Stark', 'Stephenson', 'Tazewell', 'Union',
               'Vermilion', 'Wabash', 'Warren', 'Washington', 'Wayne', 'White', 'Whiteside', 'Will', 'Williamson', 'Winnebago', 'Woodford']
counties_IN = ['Adams', 'Allen', 'Bartholomew', 'Benton', 'Blackford', 'Boone', 'Brown', 'Carroll', 'Cass', 'Clark', 'Clay', 'Clinton', 'Crawford',
               'Daviess', 'Dearborn', 'Decatur', 'DeKalb', 'Delaware', 'Dubois', 'Elkhart', 'Fayette', 'Floyd', 'Fountain', 'Franklin', 'Fulton', 'Gibson',
               'Grant', 'Greene', 'Hamilton', 'Hancock', 'Harrison', 'Hendricks', 'Henry', 'Howard', 'Huntington', 'Jackson', 'Jasper', 'Jay', 'Jefferson',
               'Jennings', 'Johnson', 'Knox', 'Kosciusko', 'LaGrange', 'Lake', 'LaPorte', 'Lawrence', 'Madison', 'Marion', 'Marshall', 'Martin', 'Miami',
               'Monroe', 'Montgomery', 'Morgan', 'Newton', 'Noble', 'Ohio', 'Orange', 'Owen', 'Parke', 'Perry', 'Pike', 'Porter', 'Posey', 'Pulaski',
               'Putnam', 'Randolph', 'Ripley', 'Rush', 'St. Joseph', 'Scott', 'Shelby', 'Spencer', 'Starke', 'Stueben', 'Sullivan', 'Switzerland',
               'Tippecanoe', 'Tipton', 'Union', 'Vanderburgh', 'Vermillion', 'Vigo', 'Wabash', 'Warren', 'Warrick', 'Washington', 'Wayne', 'Wells', 'White',
               'Whitley']
counties_IA = ['Adair', 'Adams', 'Allamakee', 'Appanoose', 'Audubon', 'Benton', 'Black Hawk', 'Boone', 'Bremer', 'Buchanan', 'Buena Vista', 'Butler',
               'Calhoun', 'Carroll', 'Cass', 'Cedar', 'Cerro Gordo', 'Cherokee', 'Chickasaw', 'Clarke', 'Clay', 'Clayton', 'Clinton', 'Crawford', 'Dallas',
               'Davis', 'Decatur', 'Delaware', 'Des Moines', 'Dickinson', 'Dubuque', 'Emmet', 'Fayette', 'Floyd', 'Franklin', 'Fremont', 'Greene',
               'Grundy', 'Guthrie', 'Hamilton', 'Hancock', 'Hardin', 'Harrison', 'Henry', 'Howard', 'Humboldt', 'Ida', 'Iowa', 'Jackson', 'Jasper',
               'Jefferson', 'Johnson', 'Jones', 'Keokuk', 'Kossuth', 'Lee', 'Linn', 'Louisa', 'Lucas', 'Lyon', 'Madison', 'Mahaska', 'Marion', 'Marshall',
               'Mills', 'Mitchell', 'Monona', 'Monroe', 'Montgomery', 'Muscatine', 'O\'Brien', 'Osceola', 'Osceola', 'Palo Alto', 'Plymouth',
               'Pocahontas', 'Polk', 'Pottawattamie', 'Powesheik', 'Ringgold', 'Sac', 'Scott', 'Shelby', 'Sioux', 'Story', 'Tama', 'Taylor', 'Union',
               'Van Buren', 'Wapello', 'Warren', 'Washington', 'Wayne', 'Webster', 'Winnebago', 'Winneshiek', 'Woodbury', 'Worth', 'Wright']
counties_KS = ['Allen', 'Anderson', 'Atchison', 'Barber', 'Barton', 'Bourbon', 'Brown', 'Butler', 'Chase', 'Chautauqua', 'Cherokee', 'Cheyenne', 'Clark',
               'Clay', 'Cloud', 'Coffey', 'Comanche', 'Cowley', 'Crawford', 'Decatur', 'Dickinson', 'Doniphan', 'Douglas', 'Edwards', 'Elk', 'Ellis',
               'Ellsworth', 'Finney', 'Ford', 'Franklin', 'Geary', 'Gove', 'Graham', 'Grant', 'Gray', 'Greeley', 'Greenwood', 'Hamilton', 'Harper',
               'Harvey', 'Haskell', 'Hodgeman', 'Jackson', 'Jefferson', 'Jewell', 'Johnson', 'Kearny', 'Kingman', 'Kiowa', 'Labette', 'Lane',
               'Leavenworth', 'Lincoln', 'Linn', 'Logan', 'Lyon', 'Marion', 'Marshall', 'McPherson', 'Meade', 'Miami', 'Mitchell', 'Montgomery',
               'Morris', 'Morton', 'Nemaha', 'Neosho', 'Ness', 'Norton', 'Osage', 'Osborne', 'Ottawa', 'Pawnee', 'Phillips', 'Pottawatomie', 'Pratt',
               'Rawlins', 'Reno', 'Republic', 'Rice', 'Riley', 'Rooks', 'Rush', 'Russell', 'Saline', 'Scott', 'Sedgwick', 'Seward', 'Shawnee', 'Sheridan',
               'Sherman', 'Smith', 'Stafford', 'Stanton', 'Stevens', 'Sumner', 'Thomas', 'Trego', 'Wabunsee', 'Wallace', 'Washington', 'Wichita', 'Wilson',
               'Woodson', 'Wyandotte']
counties_KY = ['Adair', 'Allen', 'Anderson', 'Ballard', 'Barren', 'Bath', 'Bell', 'Boone', 'Bourbon', 'Boyd', 'Boyle', 'Bracken', 'Breathitt',
               'Breckinridge', 'Bullitt', 'Butler', 'Caldwell', 'Calloway', 'Campbell', 'Carlisle', 'Carroll', 'Carter', 'Casey', 'Christian', 'Clark',
               'Clay', 'Clinton', 'Crittenden', 'Cumberland', 'Daviess', 'Edmonson', 'Elliott', 'Estill', 'Fayette', 'Fleming', 'Floyd', 'Franklin',
               'Fulton', 'Gallatin', 'Garrard', 'Grant', 'Graves', 'Grayson', 'Green', 'Greenup', 'Hancock', 'Hardin', 'Harlan', 'Harrison', 'Hart',
               'Henderson', 'Henry', 'Hickman', 'Hopkins', 'Jackson', 'Jefferson', 'Jessamine', 'Johnson', 'Kenton', 'Knott', 'Knox', 'LaRue', 'Laurel',
               'Lawrence', 'Lee', 'Leslie', 'Letcher', 'Lewis', 'Lincoln', 'Livingston', 'Logan', 'Lyon', 'Madison', 'Magoffin', 'Marion', 'Marshall',
               'Martin', 'Mason', 'McCracken', 'McCreary', 'McLean', 'Meade', 'Menifee', 'Mercer', 'Matcalfe', 'Monroe', 'Montgomery', 'Morgan',
               'Muhlenburg', 'Nelson', 'Nicholas', 'Ohio', 'Oldham', 'Owen', 'Owsley', 'Pendleton', 'Perry', 'Pike', 'Powell', 'Pulaski', 'Robertson',
               'Rockcastle', 'Rowan', 'Russell', 'Scott', 'Shelby', 'Simpson', 'Spencer', 'Taylor', 'Todd', 'Trigg', 'Trimble', 'Union', 'Warren',
               'Washington', 'Wayne', 'Webster', 'Whitley', 'Wolfe', 'Woodford']
counties_LA = ['Acadia', 'Allen', 'Ascension', 'Assumption', 'Avoyelles', 'Beauregard', 'Bienville', 'Bossier', 'Caddo', 'Calcasieu', 'Caldwell',
               'Cameron', 'Catahoula', 'Claiborne', 'Concordia', 'DeSoto', 'East Baton Rouge', 'East Carroll', 'East Feliciana', 'Evangeline',
               'Franklin', 'Grant', 'Iberia', 'Iberville', 'Jackson', 'Jefferson', 'Jefferson Davis', 'Lafayette', 'Lafourche', 'LaSalle', 'Lincoln',
               'Livingston', 'Madison', 'Morehouse', 'Natchitoches', 'Orleans', 'Ouachita', 'Plaquemines', 'Pointe Coupee', 'Rapides', 'Red River',
               'Richland', 'Sabine', 'St. Bernard', 'St. Charles', 'St. Helena', 'St. James', 'St. John the Baptist', 'St. Landry', 'St. Martin',
               'St. Mary', 'St. Tammany', 'Tangipahoa', 'Tensas', 'Terrebone', 'Union', 'Vermilion', 'Vernon', 'Washington', 'Webster',
               'West Baton Rouge', 'West Carroll', 'West Feliciana', 'Winn']
counties_ME = ['Androscoggin', 'Aroostook', 'Cumberland', 'Franklin', 'Hancock', 'Kennebec', 'Knox', 'Lincoln', 'Oxford', 'Penobscot', 'Piscataquis',
               'Sagasahoc', 'Somerset', 'Waldo', 'Washington', 'York']
counties_MD = ['Allegany', 'Anne Arundel', 'Baltimore', 'Baltimore City', 'Calvert', 'Caroline', 'Carroll', 'Cecil', 'Charles', 'Dorchester', 'Frederick',
               'Garrett', 'Harford', 'Howard', 'Kent', 'Montgomery', 'Prince George’s', 'Queen Anne’s', 'St. Mary’s', 'Somerset', 'Talbot', 'Washington',
               'Wicomico', 'Worcester']
counties_MA = ['Barnstable', 'Berkshire', 'Bristol', 'Dukes', 'Essex', 'Franklin', 'Hampden', 'Hampshire', 'Middlesex', 'Nantucket', 'Norfolk', 'Plymouth',
               'Suffolk', 'Worcester']
counties_MI = ['Alcona', 'Alger', 'Allegan', 'Alpena', 'Antrim', 'Arenac', 'Baraga', 'Barry', 'Bay', 'Benzie', 'Berrien', 'Branch', 'Calhoun', 'Cass',
               'Charlevoix', 'Cheboygan', 'Chippewa', 'Clare', 'Crawford', 'Crawford', 'Delta', 'Dickinson', 'Eaton', 'Emmet', 'Genesee', 'Gladwin',
               'Gogebic', 'Grand Traverse', 'Gratiot', 'Hillsdale', 'Houghton', 'Huron', 'Ingham', 'Ionia', 'Iosco', 'Irom', 'Isabella', 'Jackson',
               'Kalamazoo', 'Kalkaska', 'Kent', 'Keweenaw', 'Lake', 'Lapeer', 'Leelanau', 'Lenawee', 'Livingston', 'Luce', 'Mackinac', 'Macomb',
               'Manistee', 'Marquette', 'Mason', 'Mecosta', 'Menominee', 'Midland', 'Missaukee', 'Monroe', 'Montcalm', 'Montmorency', 'Muskegon',
               'Newaygo', 'Oakland', 'Oceana', 'Ogemaw', 'Ontonagon', 'Osceola', 'Oscoda', 'Otsego', 'Ottawa', 'Presque Isle', 'Roscommon', 'Saginaw',
               'St. Clair', 'St. Joseph', 'Sanilac', 'Schoolcraft', 'Shiawassee', 'Tuscola', 'Van Buren', 'Washtenaw', 'Wayne', 'Wexford']
counties_MN = ['Aitkin', 'Anoka', 'Becker', 'Beltrami', 'Benton', 'Big Stone', 'Blue Earth', 'Brown', 'Carlton', 'Carver', 'Cass', 'Chippewa', 'Chisago',
               'Clay', 'Clearwater', 'Cook', 'Cottonwood', 'Crow Wing', 'Dakota', 'Dodge', 'Douglas', 'Faribault', 'Fillmore', 'Freeborn', 'Goodhue',
               'Grant', 'Hennepin', 'Houston', 'Hubbard', 'Isanti', 'Itasca', 'Jackson', 'Kanabec', 'Kandiyohi', 'Kittson', 'Koochiching', 'Lac qui Parle',
               'Lake', 'Lake of the Woods', 'LeSueur', 'Lincoln', 'Lyon', 'Mahnomen', 'Marshall', 'Martin', 'McLeod', 'Meeker', 'Mille Lacs', 'Morrison',
               'Mower', 'Murray', 'Nicollet', 'Nobles', 'Norman', 'Olmsted', 'Otter Tail', 'Pennington', 'Pine', 'Pipestone', 'Polk', 'Pope', 'Ramsey',
               'Red Lake', 'Redwood', 'Renville', 'Rice', 'Rock', 'Roseau', 'St. Louis', 'Scott', 'Sherburne', 'Sibley', 'Stearns', 'Steele', 'Stevens',
               'Swift', 'Todd', 'Traverse', 'Wabasha', 'Wadena', 'Waseca', 'Washington', 'Watonwan', 'Wilkin', 'Winona', 'Wright', 'Yellow Medicine']
counties_MS = ['Adams', 'Alcorn', 'Amite', 'Attala', 'Benton', 'Bolivar', 'Calhoun', 'Carroll', 'Chickasaw', 'Choctaw', 'Claiborne', 'Clarke', 'Clay',
               'Coahoma', 'Copiah', 'Covington', 'DeSoto', 'Forrest', 'Franklin', 'George', 'Greene', 'Grenada', 'Hancock', 'Harrison', 'Hinds', 'Holmes',
               'Humphreys', 'Issaquena', 'Itawamba', 'Jackson', 'Jasper', 'Jefferson', 'Jefferson Davis', 'Jones', 'Kemper', 'Lafayette', 'Lamar',
               'Lauderdale', 'Lawrence', 'Leake', 'Lee', 'Leflore', 'Lincoln', 'Lowndes', 'Madison', 'Marion', 'Marshall', 'Monroe', 'Montgomery',
               'Neshoba', 'Newton', 'Noxubee', 'Oktibbeha', 'Panola', 'Pearl River', 'Perry', 'Pike', 'Pontotoc', 'Prentiss', 'Quitman', 'Rankin', 'Scott',
               'Sharkey', 'Simpson', 'Smith', 'Stone', 'Sunflower', 'Tallahatchie', 'Tate', 'Tippah', 'Tishomingo', 'Tunica', 'Union', 'Walthall',
               'Warren', 'Washington', 'Wayne', 'Webster', 'Wilkinson', 'Winston', 'Yalobusha', 'Yazoo']
counties_MO = ['Adair', 'Andrew', 'Atchison', 'Audrain', 'Barry', 'Barton', 'Bates', 'Benton', 'Bollinger', 'Boone', 'Buchanan', 'Butler', 'Caldwell',
               'Callaway', 'Camden', 'Cape Girardeau', 'Carroll', 'Carter', 'Cass', 'Cedar', 'Chariton', 'Christian', 'Clark', 'Clay', 'Clinton', 'Cole',
               'Cooper', 'Crawford', 'Dade', 'Dallas', 'Daviess', 'DeKalb', 'Dent', 'Douglas', 'Dunklin', 'Franklin', 'Gasconade', 'Gentry', 'Greene',
               'Grundy', 'Harrison', 'Henry', 'Hickory', 'Holt', 'Howard', 'Howell', 'Iron', 'Jackson', 'Jasper', 'Jefferson', 'Johnson', 'Knox',
               'Laclede', 'Lafayette', 'Lawrence', 'Lewis', 'Lincoln', 'Linn', 'Livingston', 'Macon', 'Madison', 'Maries', 'Marion', 'McDonald', 'Mercer',
               'Miller', 'Mississippi', 'Moniteau', 'Monroe', 'Montgomery', 'Morgan', 'New Madrid', 'Newton', 'Nodaway', 'Oregon', 'Osage', 'Ozark',
               'Pemiscot', 'Perry', 'Pettis', 'Phelps', 'Pike', 'Plater', 'Polk', 'Pulaski', 'Putnam', 'Ralls', 'Randolph', 'Ray', 'Reynolds', 'Ripley',
               'St. Charles', 'St. Clair', 'St. Francis', 'St. Louis', 'St. Louis City', 'Ste. Genevieve', 'Saline', 'Schuyler', 'Scotland', 'Scott',
               'Shannon', 'Shelby', 'Stoddard', 'Stone', 'Sullivan', 'Taney', 'Texas', 'Vernon', 'Warren', 'Washington', 'Wayne', 'Webster', 'Worth',
               'Wright']
counties_MT = ['Beaverhead', 'Big Horn', 'Blaine', 'Broadwater', 'Carbon', 'Carter', 'Cascade', 'Chouteau', 'Custer', 'Daniel', 'Dawson', 'Deer Lodge',
               'Fallon', 'Fergus', 'Flathead', 'Gallatin', 'Garfield', 'Glacier', 'Golden Valley', 'Granite', 'Hill', 'Jefferson', 'Judith Basin', 'Lake',
               'Lewis and Clark', 'Liberty', 'Lincoln', 'Madison', 'McCone', 'Meagher', 'Mineral', 'Missoula', 'Musselshell', 'Park', 'Petroleum',
               'Phillips', 'Pondera', 'Powder River', 'Powell', 'Prairie', 'Ravali', 'Richland', 'Roosevelt', 'Rosebud', 'Sanders', 'Sheridan',
               'Silver Bow', 'Stillwater', 'Sweet Grass', 'Teton', 'Toole', 'Treasure', 'Valley', 'Wheatland', 'Wibaux', 'Yellowstone']
counties_NE = ['Adams', 'Antelope', 'Arthur', 'Banner', 'Blaine', 'Boone', 'Box Butte', 'Boyd', 'Brown', 'Buffalo', 'Burt', 'Butler', 'Cass', 'Cedar',
               'Chase', 'Cherry', 'Cheyenne', 'Clay', 'Colfax', 'Cuming', 'Custer', 'Dakota', 'Dawes', 'Dawson', 'Deuel', 'Dixon', 'Dodge', 'Douglas',
               'Dundy', 'Fillmore', 'Franklin', 'Frontier', 'Furnas', 'Gage', 'Garden', 'Garfield', 'Gosper', 'Grant', 'Greeley', 'Hall', 'Hamilton',
               'Harlan', 'Hayes', 'Hitchcock', 'Holt', 'Hooked', 'Howard', 'Jefferson', 'Johnson', 'Kearney', 'Keith', 'Keya Paha', 'Kimball', 'Knox',
               'Lancaster', 'Lincoln', 'Logan', 'Loup', 'Madison', 'McPherson', 'Merrick', 'Morrill', 'Nance', 'Nehama', 'Nuckolls', 'Otoe', 'Pawnee',
               'Perkins', 'Phelps', 'Pierce', 'Platte', 'Polk', 'Red Willow', 'Richardson', 'Rock', 'Saline', 'Sarpy', 'Saunders', 'Scotts Bluff',
               'Seward', 'Sheridan', 'Sherman', 'Sioux', 'Stanton', 'Thayer', 'Thomas', 'Thurston', 'Valley', 'Washington', 'Wayne', 'Webster', 'Wheeler',
               'York']
counties_NV = ['Carson City', 'Churchill', 'Clark', 'Douglas', 'Elko', 'Esmeralda', 'Eureka', 'Humboldt', 'Lander', 'Lincoln', 'Lyon', 'Mineral', 'Nye',
               'Pershing', 'Storey', 'Washoe', 'White Pine']
counties_NH = ['Belknap', 'Carroll', 'Cheshire', 'Coos', 'Grafton', 'Hillsborough', 'Merrimack', 'Rockingham', 'Strafford', 'Sullivan']
counties_NJ = ['Atlantic', 'Bergen', 'Burlington', 'Camden', 'Cape May', 'Cumberland', 'Essex', 'Gloucester', 'Hudson', 'Hunterdon', 'Mercer', 'Middlesex',
               'Monmouth', 'Morris', 'Ocean', 'Passaic', 'Salem', 'Somerset', 'Sussex', 'Union', 'Warren']
counties_NM = ['Bernalillo', 'Catron', 'Chaves', 'Cibola', 'Colfax', 'Curry', 'De Baca', 'Doña Ana', 'Eddy', 'Grant', 'Guadalupe', 'Harding', 'Hidalgo',
               'Lea', 'Lincoln', 'Los Alamos', 'Luna', 'McKinley', 'Mora', 'Otero', 'Quay', 'Rio Arriba', 'Roosevelt', 'Sandoval', 'San Juan',
               'San Miguel', 'Santa Fe', 'Sierra', 'Socorro', 'Taos', 'Torrance', 'Union', 'Valencia']
counties_NY = ['Albany', 'Allegany', 'Bronx', 'Broome', 'Cattaraugus', 'Cayuga', 'Chautauqua', 'Chemung', 'Chenago', 'Clinton', 'Columbia', 'Cortland',
               'Delaware', 'Dutchess', 'Erie', 'Essex', 'Franklin', 'Fulton', 'Genesee', 'Greene', 'Hamilton', 'Herkimer', 'Jefferson', 'Kings', 'Lewis',
               'Livingston', 'Madison', 'Monroe', 'Montgomery', 'Nassau', 'New York', 'Niagara', 'Oneida', 'Onondaga', 'Ontario', 'Orange', 'Orleans',
               'Oswego', 'Otsego', 'Putnam', 'Queens', 'Rensselaer', 'Richmond', 'Rockland', 'St. Lawrence', 'Saratoga', 'Schenectady', 'Schoharie',
               'Schuyler', 'Seneca', 'Steuben', 'Suffolk', 'Sullivan', 'Tioga', 'Tompkins', 'Ulster', 'Warren', 'Washington', 'Wayne', 'Westchester',
               'Wyoming', 'Yates']
counties_NC = ['Alamance', 'Alexander', 'Alleghany', 'Anson', 'Ashe', 'Avery', 'Beaufort', 'Bertie', 'Bladen', 'Brunswick', 'Buncombe', 'Burke',
               'Cabarrus', 'Caldwell', 'Camden', 'Carteret', 'Caswell', 'Catawba', 'Chatham', 'Cherokee', 'Chowan', 'Clay', 'Cleveland', 'Columbus',
               'Craven', 'Cumberland', 'Currituck', 'Dare', 'Davidson', 'Davie', 'Duplin', 'Durham', 'Edgecombe', 'Forsyth', 'Franklin', 'Gaston','Gates',
               'Graham', 'Granville', 'Greene', 'Guilford', 'Halifax', 'Harnett', 'Haywood', 'Henderson', 'Hertford', 'Hoke', 'Hyde', 'Iredell', 'Jackson',
               'Johnston', 'Jones', 'Lee', 'Lenoir', 'Lincoln', 'Macon', 'Madison', 'Martin', 'McDowell', 'Mecklenburg', 'Mitchell', 'Montgomery', 'Moore',
               'Nash', 'New Hanover', 'Northampton', 'Onslow', 'Orange', 'Pamlico', 'Pasquotank', 'Pender', 'Perquimans', 'Person', 'Pitt', 'Polk',
               'Randolph', 'Richmond', 'Robeson', 'Rockingham', 'Rowan', 'Rutherford', 'Sampson', 'Scotland', 'Stanly', 'Stokes', 'Surry', 'Swain',
               'Transylvania', 'Tyrrell', 'Union', 'Vance', 'Wake', 'Warren', 'Washington', 'Watauga', 'Wayne', 'Wilkes', 'Wilson', 'Yadkin', 'Yancey']
counties_ND = ['Adams', 'Barnes', 'Benson', 'Billings', 'Bottineau', 'Bowman', 'Burke', 'Burleigh', 'Cass', 'Cavalier', 'Dickey', 'Divide', 'Dunn', 'Eddy',
               'Emmons', 'Foster', 'Golden Valley', 'Grand Forks', 'Grant', 'Griggs', 'Hettinger', 'Kidder', 'LaMoure', 'Logan', 'McHenry', 'McIntosh',
               'McKenzie', 'McLean', 'Mercer', 'Morton', 'Mountrail', 'Nelson', 'Oliver', 'Pembina', 'Pierce', 'Ramsey', 'Ransom', 'Renville', 'Richland',
               'Rolette', 'Sargent', 'Sheridan', 'Sioux', 'Slope', 'Stark', 'Steele', 'Stutsman', 'Towner', 'Traill', 'Walsh', 'Ward', 'Wells', 'Williams']
counties_OH = ['Adams', 'Allen', 'Ashland', 'Ashtabula', 'Athens', 'Auglaize', 'Belmont', 'Brown', 'Butler', 'Carroll', 'Champaign', 'Clark', 'Clermont',
               'Clinton', 'Columbiana', 'Coshocton', 'Crawford', 'Cuyahoga', 'Darke', 'Defiance', 'Delaware', 'Erie', 'Fairfield', 'Fayette', 'Franklin',
               'Fulton', 'Gallia', 'Geauga', 'Greene', 'Guernsey', 'Hamilton', 'Hancock', 'Hardin', 'Harrison', 'Henry', 'Highland', 'Hocking', 'Holmes',
               'Huron', 'Jackson', 'Jefferson', 'Knox', 'Lake', 'Lawrence', 'Licking', 'Logan', 'Lorain', 'Lucas', 'Madison', 'Mahoning', 'Marion',
               'Medina', 'Meigs', 'Mercer', 'Miami', 'Monroe', 'Montgomery', 'Morgan', 'Morrow', 'Muskingum', 'Noble', 'Ottawa', 'Paulding', 'Perry',
               'Pickaway', 'Pike', 'Portage', 'Preble', 'Putnam', 'Richland', 'Ross', 'Sandusky', 'Scioto', 'Seneca', 'Shelby', 'Stark', 'Summit',
               'Trumbull', 'Tuscarawas', 'Union', 'Van Wert', 'Vinton', 'Warren', 'Washington', 'Wayne', 'Williams', 'Wood', 'Wyandot']
counties_OK = ['Adair', 'Alfalfa', 'Atoka', 'Beaver', 'Beckham', 'Blaine', 'Bryan', 'Caddo', 'Canadian', 'Carter', 'Cherokee', 'Choctaw', 'Cimarron',
               'Cleveland', 'Coal', 'Comanche', 'Cotton', 'Craig', 'Creek', 'Custer', 'Delaware', 'Dewey', 'Ellis', 'Garfield', 'Garvin', 'Grady',
               'Grant', 'Greer', 'Harmon', 'Harper', 'Haskell', 'Hughes', 'Jackson', 'Jefferson', 'Johnston', 'Kay', 'Kingfisher', 'Kiowa', 'Latimer',
               'LeFlore', 'Lincoln', 'Logan', 'Love', 'Major', 'Marshall', 'Mayes', 'McClain', 'McCurtain', 'McIntosh', 'Murray', 'Muskogee', 'Noble',
               'Nowata', 'Okfuskee', 'Oklahoma', 'Okmulgee', 'Osage', 'Ottawa', 'Payne', 'Payne', 'Pittsburg', 'Pontotoc', 'Pottawatomie', 'Pushmataha',
               'Roger Mills', 'Rogers', 'Seminole', 'Sequoyah', 'Stephens', 'Texas', 'Tillman', 'Tulsa', 'Wagoner', 'Washington', 'Washita', 'Woods',
               'Woodward']
counties_OR = ['Baker', 'Benton', 'Clackamas', 'Clatsop', 'Columbia', 'Coos', 'Crook', 'Curry', 'Deschutes', 'Douglas', 'Gilliam', 'Grant', 'Harney',
               'Hood River', 'Jackson', 'Jefferson', 'Josephine', 'Klamath', 'Lake', 'Lane', 'Lincoln', 'Linn', 'Malheur', 'Marion', 'Morrow', 'Multnomah',
               'Polk', 'Sherman', 'Tillamook', 'Umatilla', 'Union', 'Wallowa', 'Wasco', 'Washington', 'Wheeler', 'Yamhill']
counties_PA = ['Adams', 'Alleghany', 'Armstrong', 'Beaver', 'Bedford', 'Berks', 'Blair', 'Bradford', 'Bucks', 'Butler', 'Cambria', 'Cameron', 'Carbon',
               'Centre', 'Chester', 'Clarion', 'Clearfield', 'Clinton', 'Columbia', 'Crawford', 'Cumberland', 'Dauphin', 'Delaware', 'Elk', 'Erie',
               'Fayette', 'Forest', 'Franklin', 'Fulton', 'Greene', 'Huntingdon', 'Indiana', 'Jefferson', 'Juniata', 'Lackawanna', 'Lancaster', 'Lawrence',
               'Lebanon', 'Lehigh', 'Luzerne', 'Lycoming', 'McKean', 'Mercer', 'Mifflin', 'Monroe', 'Montgomery', 'Montour', 'Northampton',
               'Northumberland', 'Perry', 'Philadelphia', 'Pike', 'Potter', 'Schuylkill', 'Synder', 'Somerset', 'Sullivan', 'Susquehanna', 'Tioga', 'Union',
               'Venango', 'Warren', 'Washington', 'Wayne', 'Westmoreland', 'Wyoming', 'York']
counties_RI = ['Bristol', 'Kent', 'Newport', 'Providence', 'Washington']
counties_SC = ['Abbeville', 'Aiken', 'Allendale', 'Anderson', 'Bamberg', 'Barnwell', 'Beaufort', 'Berkeley', 'Calhoun', 'Charleston', 'Cherokee',
               'Chester', 'Chesterfield', 'Clarendon', 'Colleton', 'Darlington', 'Dillon', 'Dorchester', 'Edgefield', 'Fairfield', 'Florence',
               'Georgetown', 'Greenville', 'Greenwood', 'Hampton', 'Horry', 'Jasper', 'Kershaw', 'Lancaster', 'Laurens', 'Lee', 'Lexington', 'Marion',
               'Marlboro', 'McCormick', 'Newberry', 'Oconee', 'Orangeburg', 'Pickens', 'Richland', 'Saluda', 'Spartanburg', 'Sumter', 'Union',
               'Williamsburg', 'York']
counties_SD = ['Aurora', 'Beadle', 'Bennett', 'Bon Homme', 'Brookings', 'Brown', 'Brule', 'Buffalo', 'Butte', 'Campbell', 'Charles Mix', 'Clark', 'Clay',
               'Codington', 'Corson', 'Custer', 'Davison', 'Day', 'Deuel', 'Dewey', 'Douglas', 'Edmunds', 'Fall River', 'Faulk', 'Grant', 'Gregory',
               'Haakon', 'Hamlin', 'Hand', 'Hanson', 'Harding', 'Hughes', 'Hutchinson', 'Hyde', 'Jackson', 'Jerauld', 'Jones', 'Kingsbury', 'Lake',
               'Lawrence', 'Lincoln', 'Lyman', 'Marshall', 'McCook', 'McPherson', 'Meade', 'Mellette', 'Miner', 'Minnehaha', 'Moody', 'Oglala Lakota',
               'Pennington', 'Perkins', 'Potter', 'Roberts', 'Sanborn', 'Spink', 'Stanley', 'Sully', 'Todd', 'Tripp', 'Turner', 'Union', 'Walworth',
               'Yankton', 'Ziebach']
counties_TN = ['Anderson', 'Bedford', 'Benton', 'Bledsoe', 'Blount', 'Bradley', 'Campbell', 'Cannon', 'Carroll', 'Carter', 'Cheatham', 'Chester',
               'Claiborne', 'Clay', 'Cocke', 'Coffee', 'Crockett', 'Cumberland', 'Davidson', 'Decatur', 'DeKalb', 'Dickson', 'Dyer', 'Fayette', 'Fentress',
               'Franklin', 'Gibson', 'Giles', 'Grainger', 'Greene', 'Grundy', 'Hamblen', 'Hamilton', 'Hancock', 'Hardeman', 'Hardin', 'Hawkins', 'Haywood',
               'Henderson', 'Henry', 'Hickman', 'Houston', 'Humphreys', 'Jackson', 'Jefferson', 'Johnson', 'Knox', 'Lake', 'Lauderdale', 'Lawrence',
               'Lewis', 'Lincoln', 'Loudoj', 'Macon', 'Madison', 'Marion', 'Marshall', 'Maury', 'McMinn', 'McNairy', 'Meigs', 'Monroe', 'Montgomery',
               'Moore', 'Morgan', 'Obion', 'Overton', 'Perry', 'Pickett', 'Polk', 'Putnam', 'Rhea', 'Roane', 'Robertson', 'Rutherford', 'Scott',
               'Sequatchie', 'Sevier', 'Shelby', 'Smith', 'Stewart', 'Sullivan', 'Sumner', 'Tipton', 'Trousdale', 'Unicoi', 'Union', 'Van Buren', 'Warren',
               'Washington', 'Wayne', 'Weakley', 'White', 'Williamson', 'Wilson']
counties_TX = [' Anderson', 'Andrews', 'Angelina', 'Aransas', 'Archer', 'Armstrong', 'Atascosa', 'Austin', 'Bailey', 'Bandera', 'Bastrop', 'Baylor', 'Bee',
               'Bell', 'Bexar', 'Blanco', 'Borden', 'Bosque', 'Bowie', 'Brazoria', 'Brazos', 'Brewster', 'Briscoe', 'Brooks', 'Brown', 'Burleson',
               'Burnet', 'Caldwell', 'Calhoun', 'Callahan', 'Cameron', 'Camp', 'Carson', 'Cass', 'Castro', 'Chambers', 'Cherokee', 'Childress', 'Clay',
               'Cochran', 'Coke', 'Coleman', 'Collin', 'Collingsworth', 'Colorado', 'Comal', 'Comanche', 'Concho', 'Cooke', 'Coryell', 'Cottle', 'Crane',
               'Crockett', 'Crosby', 'Culberson', 'Dallam', 'Dallas', 'Dawson', 'Deaf Smith', 'Delta', 'Denton', 'DeWitt', 'Dickens', 'Dimmit', 'Donley',
               'Duval', 'Eastland', 'Ector', 'Edwards', 'Ellis', 'El Paso', 'Erath', 'Falls', 'Fannin', 'Fayette', 'Fisher', 'Floyd', 'Foard', 'Fort Bend',
               'Franklin', 'Freestone', 'Frio', 'Gaines', 'Galveston', 'Garza', 'Gillespie', 'Glasscock', 'Goliad', 'Gonzales', 'Gray', 'Grayson', 'Gregg',
               'Grimes', 'Guadalupe', 'Hale', 'Hall', 'Hamilton', 'Hansford', 'Hardeman', 'Hardin', 'Harris', 'Harrison', 'Hartley', 'Haskell', 'Hays',
               'Hemphill', 'Henderson', 'Hidalgo', 'Hill', 'Hockley', 'Hood', 'Hopkins', 'Houston', 'Howard', 'Hudspeth', 'Hunt', 'Hutchinson', 'Irion',
               'Jack', 'Jackson', 'Jasper', 'Jeff Davis', 'Jefferson', 'Jim Hogg', 'Jim Wells', 'Johnson', 'Jones', 'Karnes', 'Kaufman', 'Kendall',
               'Kenedy', 'Kent', 'Kerr', 'Kimble', 'King', 'Kinney', 'Kleberg', 'Knox', 'Lamar', 'Lamb', 'Lampasas', 'LaSalle', 'Lavaca', 'Lee', 'Leon',
               'Liberty', 'Limestone', 'Lipscomb', 'Live Oak', 'Llano', 'Loving', 'Lubbock', 'Lynn', 'Madison', 'Marion', 'Martin', 'Mason', 'Matagorda',
               'Maverick', 'McCulloch', 'McLennan', 'McMullen', 'Medina', 'Menard', 'Midland', 'Milam', 'Mills', 'Mitchell', 'Montague', 'Montgomery',
               'Moore', 'Morris', 'Motley', 'Nacogdoches', 'Navarro', 'Newton', 'Nolan', 'Nueces', 'Ochiltree', 'Oldham', 'Orange', 'Palo Pinto', 'Panola',
               'Parker', 'Parmer', 'Pecos', 'Polk', 'Potter', 'Presidio', 'Rains', 'Randall', 'Reagan', 'Real', 'Red River', 'Reeves', 'Refugio',
               'Roberts', 'Robertson', 'Rockwall', 'Runnels', 'Rusk', 'Sabine', 'San Augustine', 'San Jacinto', 'San Patricio', 'San Saba', 'Schleicher',
               'Scurry', 'Shackelford', 'Shelby', 'Sherman', 'Smith', 'Somervell', 'Starr', 'Stephens', 'Sterling', 'Stonewall', 'Sutton', 'Swisher',
               'Tarrant', 'Taylor', 'Terrell', 'Terry', 'Throckmorton', 'Titus', 'Tom Green', 'Travis', 'Trinity', 'Tyler', 'Upshur', 'Upton', 'Uvalde',
               'Val Verde', 'Van Zandt', 'Victoria', 'Walker', 'Waller', 'Ward', 'Washington', 'Webb', 'Wharton', 'Wheeler', 'Wichita', 'Wilbarger',
               'Willacy', 'Williamson', 'Wilson', 'Winkler', 'Wise', 'Wood', 'Yoakum', 'Young', 'Zapata', 'Zavala']
counties_UT = ['Beaver', 'Box Elder', 'Cache', 'Carbon', 'Daggett', 'Davis', 'Duchesne', 'Emery', 'Garfield', 'Grand', 'Iron', 'Juab', 'Kane', 'Millard',
               'Morgan', 'Piute', 'Rich', 'Salt Lake', 'San Juan', 'Sanpete', 'Sevier', 'Summit', 'Tooele', 'Uintah', 'Utah', 'Wasatch', 'Washington',
               'Wayne', 'Weber']
counties_VT = ['Addison', 'Bennington', 'Caledonia', 'Chittenden', 'Essex', 'Franklin', 'Grand Isle', 'Lamoille', 'Orange', 'Orleans', 'Rutland',
               'Washington', 'Windham', 'Windsor']
counties_VA = ['Accomack', 'Albemarle', 'Alleghany', 'Amelia', 'Amherst', 'Appomattox', 'Arlington', 'Augusta', 'Bath', 'Bedford', 'Bland', 'Botetourt',
               'Brunswick', 'Buchanan', 'Buckingham', 'Campbell', 'Caroline', 'Carroll', 'Charles City', 'Charlotte', 'Chesterfield', 'Clarke', 'Craig',
               'Culpeper', 'Cumberland', 'Dickenson', 'Dinwiddie', 'Essex', 'Fairfax', 'Fauquier', 'Floyd', 'Fluvanna', 'Franklin', 'Frederick', 'Giles',
               'Gloucester', 'Goochland', 'Grayson', 'Greene', 'Greensville', 'Halifax', 'Hanover', 'Henrico', 'Henry', 'Highland', 'Isle of Wight',
               'James City', 'King and Queen', 'King George', 'King William', 'Lancaster', 'Lee', 'Loudoun', 'Louisa', 'Lunenburg', 'Madison', 'Mathews',
               'Mecklenburg', 'Middlesex', 'Montgomery', 'Nelson', 'New Kent', 'Northampton', 'Northumberland', 'Nottoway', 'Orange', 'Page', 'Patrick',
               'Pittsylvania', 'Powhatan', 'Prince Edward', 'Prince George', 'Prince William', 'Pulaski', 'Rappahannock', 'Richmond', 'Roanoke',
               'Rockbridge', 'Rockingham', 'Russell', 'Scott', 'Shenandoah', 'Smyth', 'Southampton', 'Spotsylvania', 'Stafford', 'Surry', 'Sussex',
               'Tazewell', 'Warren', 'Washington', 'Westmoreland', 'Wise', 'Wythe', 'York', 'Alexandria', 'Bristol', 'Buena Vista', 'Charlottesville',
               'Chesapeake', 'Colonial Heights', 'Covington', 'Danville', 'Emporia', 'Fairfax', 'Falls Church', 'Franklin', 'Fredericksburg', 'Galax',
               'Hampton', 'Harrisonburg', 'Hopewell', 'Lexington', 'Lynchburg', 'Manassas', 'Manassas Park', 'Martinsville', 'Newport News', 'Norfolk',
               'Norton', 'Petersburg', 'Poquoson', 'Portsmouth', 'Radford', 'Richmond', 'Roanoke', 'Salem', 'Staunton', 'Suffolk', 'Virginia Beach',
               'Waynesboro', 'Williamsburg', 'Winchester']
counties_WA = ['Adams', 'Asotin', 'Benton', 'Chelan', 'Clallam', 'Clark', 'Columbia', 'Cowlitz', 'Douglas', 'Ferry', 'Franklin', 'Garfield', 'Grant',
               'Grays Harbor', 'Island', 'Jefferson', 'King', 'Kitsap', 'Kittitas', 'Klickitat', 'Lewis', 'Lincoln', 'Mason', 'Okanogan', 'Pacific',
               'Pend Oreille', 'Pierce', 'San Juan', 'Skagit', 'Skamania', 'Snohomish', 'Spokane', 'Stevens', 'Thurston', 'Wahkiakum', 'Walla Walla',
               'Whatcom', 'Whitman', 'Yakima']
counties_WV = ['Barbour', 'Berkeley', 'Boone', 'Braxton', 'Brooke', 'Cabell', 'Calhoun', 'Clay', 'Doddridge', 'Fayette', 'Gilmer', 'Grant', 'Greenbrier',
               'Hampshire', 'Hancock', 'Hardy', 'Harrison', 'Jackson', 'Jefferson', 'Kanawha', 'Lewis', 'Lincoln', 'Logan', 'Marion', 'Marshall', 'Mason',
               'McDowell', 'Mercer', 'Mineral', 'Mingo', 'Monongalia', 'Monroe', 'Morgan', 'Nicholas', 'Ohio', 'Pendleton', 'Pleasants', 'Pocahontas',
               'Preston', 'Putnam', 'Raleigh', 'Randolph', 'Ritchie', 'Roane', 'Summers', 'Taylor', 'Tucker', 'Tyler', 'Upshur', 'Wayne', 'Webster',
               'Wetzel', 'Wirt', 'Wood', 'Wyoming']
counties_WI = ['Adams', 'Ashland', 'Barron', 'Bayfield', 'Brown', 'Buffalo', 'Burnett', 'Calumet', 'Chippewa', 'Clark', 'Columbia', 'Crawford', 'Dane',
               'Dodge', 'Door', 'Douglas', 'Dunn', 'Eau Claire', 'Florence', 'Fond du Lac', 'Forest', 'Grant', 'Green', 'Green Lake', 'Iowa', 'Iron',
               'Jackson', 'Jefferson', 'Juneau', 'Kenosha', 'Kewaunee', 'La Crosse', 'Lafayette', 'Langlade', 'Lincoln', 'Manitowoc', 'Marathon',
               'Marinette', 'Marquette', 'Menominee', 'Milwaukee', 'Monroe', 'Oconto', 'Oneida', 'Outagamie', 'Ozaukee', 'Pepin', 'Pierce', 'Polk',
               'Portage', 'Price', 'Racine', 'Richland', 'Rock', 'Rusk', 'St. Croix', 'Sauk', 'Sawyer', 'Shawano', 'Sheboygan', 'Taylor', 'Trempealeau',
               'Vernon', 'Vilas', 'Walworth', 'Washburn', 'Washington', 'Waukesha', 'Waupaca', 'Waushara', 'Winnebago', 'Wood']
counties_WY = ['Albany', 'Big Horn', 'Campbell', 'Carbon', 'Converse', 'Crook', 'Fremont', 'Goshen', 'Hot Springs', 'Johnson', 'Laramie', 'Lincoln',
               'Natrona', 'Niobrara', 'Park', 'Platte', 'Sheridan', 'Sublette', 'Sweetwater', 'Teton', 'Uinta', 'Washakie', 'Weston']

In [123]:
counties_of_concern = ['Alabama', counties_AL, 'Arizona', counties_AZ, 'Arkansas', counties_AR, 'California', counties_CA, 'Colorado', counties_CO,
                       'Connecticut', counties_CT, 'Delaware', counties_DE, 'Florida', counties_FL, 'Georgia', counties_GA, 'Idaho', counties_ID,
                       'Illinois', counties_IL, 'Indiana', counties_IN, 'Iowa', counties_IA, 'Kansas', counties_KS, 'Kentucky', counties_KY, 'Louisiana',
                       counties_LA, 'Maine', counties_ME, 'Maryland', counties_MD, 'Massachusetts', counties_MA, 'Minnesota', counties_MN, 'Mississippi',
                       counties_MS, 'Missouri', counties_MO, 'Montana', counties_MT, 'Nebraska', counties_NE, 'Nevada', counties_NV, 'New Hampshire',
                       counties_NH, 'New Jersey', counties_NJ, 'New Mexico', counties_NM, 'New York', counties_NY, 'North Carolina', counties_NC,
                       'North Dakota', counties_ND, 'Ohio', counties_OH, 'Oklahoma', counties_OK, 'Oregon', counties_OR, 'Pennsylvania', counties_PA,
                       'Rhode Island', counties_RI, 'South Carolina', counties_SC, 'North Carolina', counties_SD, 'Tennessee', counties_TN, 'Texas',
                       counties_TX, 'Utah', counties_UT, 'Vermont', counties_VT, 'Virginia', counties_VA, 'Washington', counties_WA, 'West Virginia',
                       counties_WV, 'Wisconsin', counties_WI, 'Wyoming', counties_WY
                      ]

# full_counties = []
# states_too = []

# for i in range(0, int(len(counties_of_concern))):
#     if i % 2 == 1:
#         for j in range(0, len(counties_of_concern[i])):
#             full_counties = full_counties + [counties_of_concern[i][j]]
#         states_too = states_too + [counties_of_concern[i - 1]] * len(counties_of_concern[i])

# emptyish_frame = pd.DataFrame({})
# emptyish_frame['county'] = full_counties
# emptyish_frame['state'] = states_too

# emptyish_frame.head(20)

In [ ]:
# Takes a very long time using this method to check to see if any data from the imported file is missing
counties_not_listed = []

for i in range(0, len(emptyish_frame)):
    county_evaluate = emptyish_frame.at[i, 'county']
    state_evaluate = emptyish_frame.at[i, 'state']
    same_county = []
    for j in range(0, len(county_gdf)):
        if county_gdf.at[j, 'county'] == county_evaluate:
            same_county = same_county + [j]
    found = False
    for j in range(0, len(same_county)):
        if county_gdf.iloc[j].loc['state'] == state_evaluate:
            found = True
    if not found:
        counties_not_listed = counties_not_listed + [str('county' + ', ' + 'state')]

counties_not_listed

In [ ]:
# The following has errors; I will give up on this for now and just assume that all counties are present in the original data

for i in range(0, len(emptyish_frame)):
    county_evaluate = emptyish_frame.iloc[i].loc['county']
    state_evaluate = emptyish_frame.iloc[i].loc['state']
    original_data_this_county = county_gdf.loc[county_gdf['county'] == county_evaluate]
    original_data_this_state = county_gdf.loc[county_gdf['state'] == state_evaluate]
    not_found = True
    while not_found:
        for i in range(0, len(original_data_this_county)):
            for j in range(0, len(original_data_this_state)):
                if original_data_this_county[i].index == original_data_this_state[j].index:
                    found = False
        if not_found:
            print(county_evaluate,', ',state_evaluate,' not found in the original data')
            break
    
    # if len(county_gdf.loc[county_gdf['state'] == emptyish_frame.iloc[i].loc['state'] and county_gdf['county'] == emptyish_frame.loc['county']]) == 0:
    #     print(empytish_frame.iloc[0].loc['county'],', ',emptyish_frame.iloc[0].loc['state'],' not found in the original data')

In [125]:
# Test splitting that guarantees at least some of every year is included in training, validation, and test sets
# Have to make a separate function from pd.sample because it has the chance to underselect for certain years

def split_50_30_20(dataframe: pd.DataFrame, index_cutoff: int = 3108, years: int = 11, seed = 1) -> list:
    try: # only works if you have a 'year' column in the original and the clone
        by_year = {} # to assign the subsets of the dataframes per year
        # yearlength = {} # to assign the number of rows that exist per year
        
        for i in range(0, 11): # act for every year in the dataset; if ambiguous the end of the range would be len(pd.unique(basic_set['year']).tolist())
            # The below data adds rows of the year to the value of a dictionary where the key is the integer of the year
            by_year[i + 2014] = dataframe.loc[dataframe['year'] == i + 2014].reset_index() # makes the subset the same as the year, from 2014 to 2024
            # yearsize = 0
            
            # for j in range(0, len(basic_set)): # act for every row in the dataset; this is only necessary if I can't take the size of the year subset
            #     if basic_set.at[j, 'year'] == i + 2014: # act only if the year in the row is the same as the year in the outer loop iteration
            #         yearsize += 1 # tallies up the number of rows in the year
            # yearlength[pd.unique(basic_set['year'])[i]] = yearsize # adds data to the yearsize dictionary
            
        # Now split each year into small DataFrames and concatenate them into the three subsets for training, validation, and testing

        train_set = pd.DataFrame({'state': [], 'year': [], 'slf_pop': [], 'slfdensity': [], 'geometry': [], 'target': []})
        validation_set = pd.DataFrame({'state': [], 'year': [], 'slf_pop': [], 'slfdensity': [], 'geometry': [], 'target': []})
        test_set = pd.DataFrame({'state': [], 'year': [], 'slf_pop': [], 'slfdensity': [], 'geometry': [], 'target': []})
        # year_split_sizes = {}

        for i in range(0, years): # act for every year in the dataset, if ambiguous the end of the range would be len(pd.unique(basic_set['year']).tolist())
            current_year_size = index_cutoff
            # current_year_size = yearlength[pd.unique(basic_set['year'])[i]]
            # year_size_train = m.ceil(0.5 * current_year_size) # round up to higher training set size in case split is uneven
            # year_size_validation = m.ceil(0.3 * current_year_size) # same for validation
            # year_size_test = current_year_size - year_size_train - year_size_validation
            # year_split_sizes[pd.unique(basic_set['year'])[i]] = [year_size_train, year_size_validation,
            #                                                      current_year_size - year_size_train - year_size_validation
            #                                                     ]
            try:
                new_seed = abs(int(m.floor(seed))) # converts any numerical seed to the floor value, then an integer if it's a float, and take the A.V.
                print('Seeded')
                
            except TypeError:
                print('The seed value passed to this function is not numerical.')

            else:
                new_seed = 1
                
            finally:
                if new_seed > 1:
                    new_seed = 1 / new_seed # returns the inverse of the seed if it is greater than 1
                # combinations_train = m.factorial(current_year_size) / (m.factorial(year_size_train) * m.factorial(current_year_size - year_size_train))
                # combinations_validation = m.factorial(current_year_size) / (m.factorial(year_size_validation) * 
                #                                                             m.factorial(current_year_size - year_size_validation))
                # combinations_test = m.factorial(current_year_size) / (m.factorial(year_size_test) * m.factorial(current_year_size - year_size_test))
                # combinations_total = m.factorial(current_year_size) / ()
                all_indices = by_year[i + 2014].index.tolist()
                index_list = list(range(0, index_cutoff))
                rand.shuffle(index_list) # add new_seed?
                print('Shuffled')
                # shuffled_indices = rand.shuffle(all_indices, new_seed)
                
                if current_year_size != len(index_list):
                    print('Some rows were lost in shuffling.')

                train_indices = index_list[0 : m.ceil(0.5 * len(index_list))]
                validation_indices = index_list[m.ceil(0.5 * len(index_list)) : m.ceil(0.8 * len(index_list))]
                test_indices = index_list[m.ceil(0.8 * len(index_list)) : len(index_list)]

                # Make this year's subsets use the indices generated for their purpose
                    
                train_this_year = by_year[i + 2014].iloc[train_indices]
                print('Training for year',i + 2014,'complete')
                validation_this_year = by_year[i + 2014].iloc[validation_indices]
                print('Validation for year',i + 2014,'complete')
                test_this_year = by_year[i + 2014].iloc[test_indices]
                print('Testing for year',i + 2014,'complete')

                # Add this year's splits to the main three split datasets
                
                train_set = pd.concat([train_set, train_this_year]) # train_set.merge(train_this_year, how = 'outer')
                print('Merging train set complete')
                validation_set = pd.concat([validation_set, validation_this_year]) # validation_set.merge(validation_this_year, how = 'outer')
                print('Merging validation set complete')
                test_this_year = pd.concat([test_set, test_this_year]) # test_set.merge(test_this_year, how = 'outer')
                print('Merging test set complete')
        
    except KeyError:
        print('Could not find the year column in the dataset.')
    else: # resorts to splitting the dataset without dealing with the years; if the dataframe was passed sorting earliest to latest years, this will
          # likely ruin a the capacity for a model to train successfully when involving the time factor
        train_set = dataframe.iloc[0 : m.ceil(len(dataframe) * 0.5)]
        validation_set = dataframe.iloc[m.ceil(len(dataframe) * 0.5) : m.ceil(len(dataframe) * 0.8)]
        test_set = dataframe.iloc[m.ceil(len(dataframe) * 0.8): len(dataframe)]
    finally:
        pass
        
    return [train_set, validation_set, test_set] # might have to convert each set into a dictionary and then reconvert into DataFrames if they can't be 
                                                 # called from a list

In [127]:
# Make a preprocessor so that extra rows are preprocessed with training data

def preprocess_rf(dataframe: pd.DataFrame, index_cutoff: int = 3108, years: int = 11, split: bool = True, seed_split: int = 1) -> list:
    # X_train, X_validation, Y_train, Y_validation = train_test_split(dataframe.drop(['target','geometry'], axis = 1), dataframe['target'], test_size = 0.20, random_state = 1, shuffle = True)
    encoder = OneHotEncoder(handle_unknown = "ignore") # doing this because training set, validation set, and test set have different number of features
    defragmented_df = dataframe.copy()
    base = encoder.fit_transform(defragmented_df.drop(columns = ['year', 'target', 'geometry'], axis = 1))
    base_df = pd.DataFrame(base.toarray())
    base_df['year'] = dataframe['year'] # add year back in because it is important to the splitting process
    base_df['target'] = dataframe['target']
    if split:
        print('Preprocessing complete. Splitting...')
        return split_50_30_20(base_df, index_cutoff, years, seed_split)
    else:
        return base_df

In [ ]:
county_gdf.head()

In [15]:
general_county_gdf.head()

,county,state,land_area,water_area,year,slf_pop,slf_pop_ye,slfdensity,y_slfdnsty,target,color,geometry
0,Houston,Alabama,1501742250,4795418,2014,0,0,0.0,0.0,0,#008000,"POLYGON ((-85.712 31.197, -85.709 31.198, -85...."
1,Choctaw,Alabama,2365900084,19114321,2014,0,0,0.0,0.0,0,#008000,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4..."
2,Russell,Alabama,1660653961,15562947,2014,0,0,0.0,0.0,0,#008000,"POLYGON ((-85.435 32.318, -85.434 32.392, -85...."
3,Sussex,Delaware,2424590442,674129051,2014,0,0,0.0,0.0,0,#008000,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5..."
4,Jackson,Alabama,2792044612,126334711,2014,0,0,0.0,0.0,0,#008000,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,..."


In [129]:
def my_encoder(df: pd.DataFrame) -> pd.DataFrame:
    encoded = pd.DataFrame({})
    columns = df.columns.tolist()
    for i in range(0, len(columns)):
        this_column = columns[i]
        print('Processing the column',this_column)
        encoded_items = []
        columntype = type(df.at[0, this_column])
        
        if columntype == str:
            unique_column_items_count = len(pd.unique(df[this_column]).tolist())
            unique_column_items = []
            
            for j in range(0, len(df)):
                this_cell = df.at[j, this_column]
                
                if this_cell not in unique_column_items:
                    unique_column_items = unique_column_items + [this_cell]
                    
                this_cell_encoded = float(unique_column_items.index(this_cell)) / float(unique_column_items_count)
                encoded_items = encoded_items + [this_cell_encoded]
                
        elif columntype == int or columntype == np.int64:
            minimum = min(df[this_column].tolist())
            maximum = max(df[this_column].tolist())
            add_by_before_multiplying = 0 - minimum
            multiply_by = 1 / float(maximum - minimum)

            for j in range(0, len(df)):
                this_cell = df.at[j, this_column]
                encoded_items = encoded_items + [float(multiply_by * (add_by_before_multiplying + this_cell))]
            
        else:
            print('Column \'',this_column,'\' cannot be encoded')
            encoded_items = df[this_column]
            
        encoded[str(i)] = encoded_items # might have to convert the string column labels into integer column labels
    return encoded

In [102]:
# type(general_county_gdf.at[0, general_county_gdf.columns.tolist()[7]])

In [95]:
# min(general_county_gdf[general_county_gdf.columns.tolist()[2]].tolist())

5300262

In [117]:
# encoded_county_gdf = my_encoder(general_county_gdf)
# encoded_county_gdf.head()

Processing the column county
Processing the column state
Processing the column land_area
Processing the column water_area
Processing the column year
Processing the column slf_pop
Processing the column slf_pop_ye
Processing the column slfdensity
Column ' slfdensity ' cannot be encoded
Processing the column y_slfdnsty
Column ' y_slfdnsty ' cannot be encoded
Processing the column target
Processing the column color
Processing the column geometry
Column ' geometry ' cannot be encoded


,0,1,2,3,4,5,6,7,8,9,10,11
0,0.000000,0.000000,0.028794,0.000341,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"POLYGON ((-85.712 31.197, -85.709 31.198, -85...."
1,0.000554,0.000000,0.045421,0.001360,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4..."
2,0.001109,0.000000,0.031851,0.001107,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"POLYGON ((-85.435 32.318, -85.434 32.392, -85...."
3,0.001663,0.020833,0.046551,0.047968,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5..."
4,0.002217,0.000000,0.053621,0.008989,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,..."


In [137]:
# Add rows to the provided DataFrame containing basic geometric information but iterate by year

## 2025

data_2025 = county_gdf
data_2025['year'] = [2025] * len(county_gdf)

general_county_2025 = pd.concat([general_county_gdf, data_2025], join = 'outer')

general_county_2025 = general_county_2025.reset_index()
Y_2025_train = general_county_2025.iloc[0 : 3108 * 11]['target']
pre_encode_2025 = general_county_2025.drop(columns = ['color', 'year', 'geometry', 'slfdensity', 'y_slfdnsty'], axis = 1)
encoded_2025 = my_encoder(pre_encode_2025)
X_2025_train = encoded_2025.iloc[0 : 3108 * 11]
X_2025_predict = encoded_2025.iloc[3108 * 11 : 3108 * 12]

Processing the column index
Processing the column county
Processing the column state
Processing the column land_area
Processing the column water_area
Processing the column slf_pop
Column ' slf_pop ' cannot be encoded
Processing the column slf_pop_ye
Column ' slf_pop_ye ' cannot be encoded
Processing the column target
Column ' target ' cannot be encoded


In [35]:
# base_county_2025 = OneHotEncoder(handle_unknown = "ignore").fit_transform(gc5)
# base_county_2025_df = pd.DataFrame(base_county_2025.toarray())
# base_county_2025_df['target'] = general_county_2025['target']

# X_2025 = base_county_2025_df.iloc[len(base_county_2025_df) - 2 * 3108 : len(base_county_2025_df) - 3108].drop(columns = ['target'], axis = 1)

MemoryError: Unable to allocate 12.1 GiB for an array with shape (37296, 43495) and data type float64

In [139]:
## 2026

data_2026 = county_gdf
data_2026['year'] = [2026] * len(county_gdf)

general_county_2026 = pd.concat([general_county_2025, data_2026], join = 'outer')

general_county_2026 = general_county_2026.reset_index()
Y_2026_train = general_county_2026.iloc[0 : 3108 * 11]['target']
pre_encode_2026 = general_county_2026.drop(columns = ['color', 'year', 'geometry', 'slfdensity', 'y_slfdnsty'], axis = 1)
encoded_2026 = my_encoder(pre_encode_2026)
X_2026_train = encoded_2026.iloc[0 : 3108 * 11]
X_2026_predict = encoded_2026.iloc[3108 * 11 : 3108 * 13]

Processing the column level_0
Processing the column index
Column ' index ' cannot be encoded
Processing the column county
Processing the column state
Processing the column land_area
Processing the column water_area
Processing the column slf_pop
Column ' slf_pop ' cannot be encoded
Processing the column slf_pop_ye
Column ' slf_pop_ye ' cannot be encoded
Processing the column target
Column ' target ' cannot be encoded


In [ ]:
# general_county_2026['target'] = general_county_2026['target'].fillna(-1)
# general_county_2026 = general_county_2025.reset_index()
# general_county_2026 = general_county_2026.drop(columns = ['color', 'year', 'geometry', 'slfdensity', 'y_slfdnsty'], axis = 1)
# gc6 = general_county_2025.drop('target', axis = 1)

# base_county_2026 = OneHotEncoder(handle_unknown = "ignore").fit_transform(gc6)
# base_county_2026_df = pd.DataFrame(base_county_2026.toarray())
# base_county_2026_df['target'] = general_county_2026['target']

# X_2026 = base_county_2026_df.iloc[len(base_county_2026_df) - 3108 : len(base_county_2026_df)].drop(columns = ['target'], axis = 1)

In [13]:
# type(base_county_2026)

scipy.sparse._csr.csr_matrix

In [ ]:
# encoded_2025 = preprocess_rf(dataframe = general_county_2025, index_cutoff = 3108, years = 12, split = False, seed_split = 1)
# encoded_2025.head() # view training data

In [ ]:
# encoded_2026 = preprocess_rf(dataframe = general_county_2026, index_cutoff = 3108, years = 13, split = False, seed_split = 1)
# encoded_2026.head() # view training data

In [ ]:
# len(encoded_2025)

In [ ]:
# X_2025 = encoded_2025.iloc[len(encoded_2025) - 3108 : len(encoded_2025)].drop(columns = ['year', 'target'], axis = 1)
# X_2026 = encoded_2026.iloc[len(encoded_2026) - 2 * 3108 : len(encoded_2026)].drop(columns = ['year', 'target'], axis = 1)

In [161]:
# Bring back the seeds and results of the most performant (chosen) Random Forest and Multi-Layer Perceptron for each type of model
# No models will be trained in 2025 predictions; the 2026 prediction will only use 2014-2024 data

models = 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Deliverables/During Research/Models/'

## Random Forests

general_county_rf_path = str(models + 'original_county.sav') # was general_spread_risk_county_rf_model.sav
general_state_rf_path = str(models + 'original_state.sav') # was general_spread_risk_state_rf_model.sav
# food_county_rf_path = str(models + 'food_host_damage_risk_county_rf_model.sav')
# food_state_rf_path = str(models + 'food_host_damage_risk_state_rf_model.sav')
# fiber_county_rf_path = str(models + 'fiber_host_damage_risk_county_rf_model.sav')
# fiber_state_rf_path = str(models + 'fiber_host_damage_risk_state_rf_model.sav')
# ornamental_county_rf_path = str(models + 'ornamental_host_damage_risk_county_rf_model.sav')
# ornamental_state_rf_path = str(models + 'ornamental_host_damage_risk_state_rf_model.sav')

general_model_rf_county = pickle.load(open(general_county_rf_path, 'rb')) # 'rb' refers to the mode of opening the file, which means Read Binary
general_model_rf_state = pickle.load(open(general_state_rf_path, 'rb'))
# food_model_rf_county = pickle.load(open(food_county_rf_path, 'rb'))
# food_model_rf_state = pickle.load(open(food_state_rf_path, 'rb'))
# fiber_model_rf_county = pickle.load(open(fiber_county_rf_path, 'rb'))
# fiber_model_rf_state = pickle.load(open(fiber_state_rf_path, 'rb'))
# ornamental_model_rf_county = pickle.load(open(ornamental_county_rf_path, 'rb'))
# ornamental_model_rf_state = pickle.load(open(ornamental_state_rf_path, 'rb'))

## Multi-Layer Perceptrons

# general_county_mlp_path = str(models + 'general_spread_risk_county_mlp_model.sav')
# general_state_mlp_path = str(models + 'general_spread_risk_state_mlp_model.sav')
# food_county_mlp_path = str(models + 'food_host_damage_risk_county_mlp_model.sav')
# food_state_mlp_path = str(models + 'food_host_damage_risk_state_mlp_model.sav')
# fiber_county_mlp_path = str(models + 'fiber_host_damage_risk_county_mlp_model.sav')
# fiber_state_mlp_path = str(models + 'fiber_host_damage_risk_state_mlp_model.sav')
# ornamental_mlp_county_path = str(models + 'ornamental_host_damage_risk_county_mlp_model.sav')
# ornamental_mlp_state_path = str(models + 'ornamental_host_damage_risk_state_mlp_model.sav')

# general_model_mlp_county = pickle.load(open(general_county_mlp_path, 'rb'))
# general_model_mlp_state = pickle.load(open(general_state_mlp_path, 'rb'))
# food_model_mlp_county = pickle.load(open(food_county_mlp_path, 'rb'))
# food_model_mlp_state = pickle.load(open(food_state_mlp_path, 'rb'))
# fiber_model_mlp_county = pickle.load(open(fiber_county_mlp_path, 'rb'))
# fiber_model_mlp_state = pickle.load(open(fiber_state_mlp_path, 'rb'))
# ornamental_model_mlp_county = pickle.load(open(ornamental_county_mlp_path, 'rb'))
# ornamental_model_mlp_state = pickle.load(open(ornamental_state_mlp_path, 'rb'))

It's time to predict the target risk values for 2025 and 2026!

In [147]:
# 2025 and 2026 general spread risk retraining on all training data

general_model_rf_county_2025 = DecisionTreeClassifier()
general_model_rf_county_2025.fit(X_2025_train, Y_2025_train)

DecisionTreeClassifier()

In [149]:

general_model_rf_county_2026 = DecisionTreeClassifier()
general_model_rf_county_2026.fit(X_2025_train, Y_2026_train)

DecisionTreeClassifier()

In [155]:
Y_2025 = general_model_rf_county_2025.predict(X_2025_predict)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- class
- county
- state


In [ ]:
Y_2026 = general_model_rf_county_2026.predict(X_2026_predict)

predictions_2026 = X_2026_predict
predictions_2026['county'] = pd.concat([county_gdf['county'], county_gdf['county']], how = 'outer')
predictions_2026['state'] = pd.concat([county_gdf['state'], county_gdf['state']], how = 'outer')
predictions_2026['class'] = Y_2026

In [165]:
data_2025_conv = data_2025.apply(pd.to_numeric, errors='coerce')
data_2026_conv = data_2026.apply(pd.to_numeric, errors='coerce')

In [166]:
grfc5 = general_model_rf_county.predict(data_2025_conv)

C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\base.py:457: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(


ValueError: X has 6 features, but DecisionTreeClassifier is expecting 10163 features as input.

In [ ]:
grfs5 = general_model_rf_state.predict(data_2025_conv)

In [163]:
# 2025 general spread risk

# g_rf_c_2025 = general_model_rf_county.score(X_2025, Y_2025)
# grfc5cr = classification_report(Y_2025, dtree.predict(X_2025))
grfc5 = general_model_rf_county.predict(data_2025_conv)
# g_mlp_c_2025 = general_model_mlp_county.score(X_2025, Y_2025)
# gmlpc5 = general_model_mlp_county.predict(X_2025)
# g_rf_s_2025 = general_model_rf_state.score(X_2025, Y_2025)
grfs5 = general_model_rf_state.predict(data_2025_conv)
# g_mlp_s_2025 = general_model_mlp_state.score(X_2025, Y_2025)
# gmlps5 = general_model_mlp_state.predict(X_2025)

C:\Users\EK111\anaconda3\Lib\site-packages\sklearn\base.py:457: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(


ValueError: could not convert string to float: 'Houston'

In [ ]:
# 2025 risk to plants grown for food

food_rf_c_2025 = food_model_rf_county.score(X_2025, Y_2025)
foodrfc5 = food_model_rf_county.predict(X_2025)
food_mlp_c_2025 = food_model_mlp_county.score(X_2025, Y_2025)
foodmlpc5 = food_model_mlp_county.predict(X_2025)
food_rf_s_2025 = food_model_rf_state.score(X_2025, Y_2025)
foodrfs5 = food_model_rf_state.predict(X_2025)
food_mlp_s_2025 = food_model_mlp_state.score(X_2025, Y_2025)
foodmlps5 = food_model_mlp_state.predict(X_2025)

In [ ]:
# 2025 risk to plants grown for timber or paper

fiber_rf_c_2025 = fiber_model_rf_county.score(X_2025, Y_2025)
fiberrfc5 = fiber_model_rf_county.predict(X_2025)
fiber_mlp_c_2025 = fiber_model_mlp_county.score(X_2025, Y_2025)
fibermlpc5 = fiber_model_mlp_county.predict(X_2025)
fiber_rf_s_2025 = fiber_model_rf_state.score(X_2025, Y_2025)
fiberrfs5 = fiber_model_rf_state.predict(X_2025)
fiber_mlp_s_2025 = fiber_model_mlp_state.score(X_2025, Y_2025)
fibermlps5 = fiber_model_mlp_state.predict(X_2025)

In [ ]:
# 2025 risk to ornamental plants

ornamental_rf_c_2025 = ornamental_model_rf_county.score(X_2025, Y_2025)
orfc5 = ornamental_model_rf_county.predict(X_2025)
ornamental_mlp_c_2025 = ornamental_model_mlp_county.score(X_2025, Y_2025)
omlpc5 = ornamental_model_mlp_county.predict(X_2025)
ornamental_rf_s_2025 = ornamental_model_rf_state.score(X_2025, Y_2025)
orfs5 = ornamental_model_rf_state.predict(X_2025)
ornamental_mlp_s_2025 = ornamental_model_mlp_state.score(X_2025, Y_2025)
omlps5 = ornamental_model_mlp_state.predict(X_2025)

In [ ]:
# 2026 general spread risk

# g_rf_c_2026 = general_model_rf_county.score(X_2026, Y_2026)
grfc6 = general_model_rf_county.predict(X_2026)
# g_mlp_c_2026 = general_model_mlp_county.score(X_2026, Y_2026)
# gmlpc6 = general_model_mlp_county.predict(X_2026)
# g_rf_s_2026 = general_model_rf_state.score(X_2026, Y_2026)
grfs6 = general_model_rf_state.predict(X_2026)
# g_mlp_s_2026 = general_model_mlp_state.score(X_2026, Y_2026)
# gmlps6 = general_model_mlp_state.predict(X_2026)

In [ ]:
# 2026 risk to plants grown for food

food_rf_c_2026 = food_model_rf_county.score(X_2026, Y_2026)
foodrfc6 = food_model_rf_county.predict(X_2026)
food_mlp_c_2026 = food_model_mlp_county.score(X_2026, Y_2026)
foodmlpc6 = food_model_mlp_county.predict(X_2026)
food_rf_s_2026 = food_model_rf_state.score(X_2026, Y_2026)
foodrfs6 = food_model_rf_state.predict(X_2026)
food_mlp_s_2026 = food_model_mlp_state.score(X_2026, Y_2026)
foodmlps6 = food_model_mlp_state.predict(X_2026)

In [ ]:
# 2026 risk to plants grown for timber or paper

fiber_rf_c_2026 = fiber_model_rf_county.score(X_2026, Y_2026)
fiberrfc6 = fiber_model_rf_county.predict(X_2026)
fiber_mlp_c_2026 = fiber_model_mlp_county.score(X_2026, Y_2026)
fibermlpc6 = fiber_model_mlp_county.predict(X_2026)
fiber_rf_s_2026 = fiber_model_rf_state.score(X_2026, Y_2026)
fiberrfs6 = fiber_model_rf_state.predict(X_2026)
fiber_mlp_s_2026 = fiber_model_mlp_state.score(X_2026, Y_2026)
fibermlps6 = fiber_model_mlp_state.predict(X_2026)

In [ ]:
# 2026 risk to ornamental plants

ornamental_rf_c_2026 = ornamental_model_rf_county.score(X_2026, Y_2026)
orfc6 = .predict(X_2026) # add root
ornamental_mlp_c_2026 = ornamental_model_mlp_county.score(X_2026, Y_2026)
omlpc6 = .predict(X_2026)
ornamental_rf_s_2026 = ornamental_model_rf_state.score(X_2026, Y_2026)
orfs6 = .predict(X_2026)
ornamental_mlp_s_2026 = ornamental_model_mlp_state.score(X_2026, Y_2026)
omlps6 = .predict(X_2026)

In [ ]:
# List accuracies so they can be parsed in a print function more easily

scores = [[g_rf_c_2025, 'general spread risk model for counties using a Random Forest classifier, evaluated for 2025'],
          [g_mlp_c_2025, 'general spread risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [g_rf_s_2025, 'general spread risk model for states using a Random Forest classifier, evaluated for 2025'],
          [g_mlp_s_2025, 'general spread risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [food_rf_c_2025, 'food-growing host plant damage risk model for counties using a Random Forest classifier, evaluated for 2025'],
          [food_mlp_c_2025, 'food-growing host plant damage risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [food_rf_s_2025, 'food-growing host plant damage risk model for states using a Random Forest classifier, evaluated for 2025'],
          [food_mlp_s_2025, 'food-growing host plant damage risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [fiber_rf_c_2025, 'timber and paper-growing host plant damage risk model for counties using a Random Forest classifier, evaluated for 2025'],
          [fiber_mlp_c_2025, 'timber and paper-growing host plant damage risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [fiber_rf_s_2025, 'timber and paper-growing host plant damage risk model for states using a Random Forest classifier, evaluated for 2025'],
          [fiber_mlp_s_2025, 'timber and paper-growing host plant damage risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [ornamental_rf_c_2025, 'ornamental host plant damage risk model for counties using a Random Forest classifier, evaluated for 2025'],
          [ornamental_mlp_c_2025, 'ornamental host plant damage risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [ornamental_rf_s_2025, 'ornamental host plant damage risk model for states using a Random Forest classifier, evaluated for 2025'],
          [ornamental_mlp_s_2025, 'ornamental host plant damage risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2025'],
          [g_rf_c_2026, 'general spread risk model for counties using a Random Forest classifier, evaluated for 2026'],
          [g_mlp_c_2026, 'general spread risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [g_rf_s_2026, 'general spread risk model for states using a Random Forest classifier, evaluated for 2026'],
          [g_mlp_s_2026, 'general spread risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [food_rf_c_2026, 'food-growing host plant damage risk model for counties using a Random Forest classifier, evaluated for 2026'],
          [food_mlp_c_2026, 'food-growing host plant damage risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [food_mlp_s_2026, 'food-growing host plant damage risk model for states using a Random Forest classifier, evaluated for 2026'],
          [food_mlp_s_2026, 'food-growing host plant damage risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [fiber_rf_c_2026, 'timber and paper-growing host plant damage risk model for counties using a Random Forest classifier, evaluated for 2026'],
          [fiber_mlp_c_2026, 'timber and paper-growing host plant damage risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [fiber_rf_s_2026, 'timber and paper-growing host plant damage risk model for states using a Random Forest classifier, evaluated for 2026'],
          [fiber_mlp_s_2026, 'timber and paper-growing host plant damage risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [ornamental_rf_c_2026, 'ornamental host plant damage risk model for counties using a Random Forest classifier, evaluated for 2026'],
          [ornamental_mlp_c_2026, 'ornamental host plant damage risk model for counties using a Multi-Layer Perceptron classifier, evaluated for 2026'],
          [ornamental_rf_s_2026, 'ornamental host plant damage risk model for states using a Random Forest classifier, evaluated for 2026'],
          [ornamental_mlp_s_2026, 'ornamental host plant damage risk model for states using a Multi-Layer Perceptron classifier, evaluated for 2026']
         ]

# List the predictions so they can be parsed through a very special print function built later

predictions_only = [grfc5, gmlpc5, grfs5, gmlps5, foodrfc5, foodmlpc5, foodrfs5, foodmlps5, fiberrfc5, fibermlpc5, fiberrfs5, fibermlps5, orfc5, omlpc5,
                    orfs5, omlps5, grfc6, gmlpc6, grfs6, gmlps6, foodrfc6, foodmlpc6, foodrfs6, foodmlps6, fiberrfc6, fibermlpc6, fiberrfs6, fibermlps6,
                    orfc6, omlpc6, orfc6, omlps6]

model_results_list = [scores[i] + [predictions_only[i]] for i in scores]

In [ ]:
# Show model accuracies

print('Decimal (out of 1, or 100%) accuracies of the models according to scoring methods; keep in mind that these are EXPECTED accuracies, not true ',
      'accuracies representative of the years 2025 and 2026:\n')
for i in range(0, 32):
    print(scores[i][1],': ',scores[i][0])

In [ ]:
# Show predictions

def show_predictions(predictions):
    choice = ''
    print('You may now choose which model\'s predictions you want to see. Currently only can be shown at a time unless you run this function multiple ',
          'times simultaneously. Enter \'q\' or \'quit\' to leave this program at any time.')
    while choice.lower() != 'q' and choice.lower() != 'quit':
        print('Which predictions do you want to see?\n',
              '- \'gc\' for the predicted spotted lanternfly spread risks across the United States by COUNTY',
              '- \'gs\' for the predicted spotted lanternfly spread risks across the United States by STATE',
              '- \'fc\' for the predicted damage risks spotted lanternflies pose to food crops across the United States by COUNTY',
              '- \'fs\' for the predicted damage risks spotted lanternflies pose to food crops across the United States by STATE',
              '- \'tc\' for the predicted damage risks spotted lanternflies pose to timber and paper crops across the United States by COUNTY',
              '- \'ts\' for the predicted damage risks spotted lanternflies pose to timber and paper crops across the United States by STATE',
              '- \'oc\' for the predicted damage risks spotted lanternflies pose to ornamental plants across the United States by COUNTY',
              '- \'os\' for the predicted damage risks spotted lanternflies pose to ornamental plants across the United States by STATE',
              '- \'q\' or \'quit\' to leave\n')
        
        choice = input('Type your selection here: ')
        modeltype = choice
        c = str(choice).lower()
        
        if c != 'q' or c != 'quit':
            break
            
        elif c == 'gc' or c == 'gs' or c == 'fc' or c == 'fs' or c == 'tc' or c == 'ts' or c == 'oc' or c == 'os':
            print('For what year would you like to see predictions? (2025 or 2026)')
            choice = str(input('Type your selection here (2025 or 2026): '))
            ch = str(choice)

            if ch == '2025':
                if c == 'gc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[0]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[1]
                if c == 'gs':
                    print('Predictions from a Random Forest classifier:')
                    predictions[2]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[3]
                if c == 'fc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[4]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[5]
                if c == 'fs':
                    print('Predictions from a Random Forest classifier:')
                    predictions[6]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[7]
                if c == 'tc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[8]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[9]
                if c == 'ts':
                    print('Predictions from a Random Forest classifier:')
                    predictions[10]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[11]
                if c == 'oc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[12]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[13]
                if c == 'os':
                    print('Predictions from a Random Forest classifier:')
                    predictions[14]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[15]
                break

            elif ch == '2026':
                if c == 'gc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[16]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[17]
                if c == 'gs':
                    print('Predictions from a Random Forest classifier:')
                    predictions[18]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[19]
                if c == 'fc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[20]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[21]
                if c == 'fs':
                    print('Predictions from a Random Forest classifier:')
                    predictions[22]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[23]
                if c == 'tc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[24]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[25]
                if c == 'ts':
                    print('Predictions from a Random Forest classifier:')
                    predictions[26]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[27]
                if c == 'oc':
                    print('Predictions from a Random Forest classifier:')
                    predictions[28]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[29]
                if c == 'os':
                    print('Predictions from a Random Forest classifier:')
                    predictions[30]
                    print('Predictions from a Multi-Layer Perceptron classifier:')
                    predictions[31]
                break

            else:
                print('Could not understand the input provided. We will show the general spread risk 2025 predictions by default.\n')
                print('Predictions from a Random Forest classifier:')
                predictions[0]
                print('Predictions from a Multi-Layer Perceptron classifier:')
                predictions[1]
                break

        else:
            print('Could not understand the input provided. Please try again.')
            
    return 0

In [ ]:
# Now look through all the predictions

show_predictions(predictions_only)